1. Transfer Learning (The Broad Concept)
    - Transfer learning is the act of taking knowledge learned from one task and applying it to a different, but related, task.

    - In deep learning, training a massive neural network on a huge dataset (like ImageNet, which has millions of images) takes massive computational power and time. The network learns to recognize fundamental building blocks of the visual world: edges, curves, gradients, and basic shapes.

    - Instead of starting with a "blank brain" (random weights) for your specific project—say, classifying types of wildflowers—you download that pre-trained network. You are transferring its foundational understanding of how to "see" images into your new project.

2. Fine-Tuning (The Specific Action)
    - Fine-tuning is how you adapt that pre-trained brain to your specific task.

    - Even though the network knows how to see edges and shapes, it currently thinks the world consists of the 1,000 categories from ImageNet (dogs, cars, boats, etc.). Fine-tuning involves tweaking the network's internal connections (weights) so it stops looking for dog ears and starts looking for specific petal shapes.

3. As Yosinski proved, we don't treat the whole network the same way during fine-tuning:

    - Early layers are kept locked (frozen) because basic edge detection is always useful.

    - Later layers are tweaked gently (fine-tuned) to shift their focus from general objects to your specific domain.

    - The final layer (the classifier) is completely chopped off, replaced with a brand new one, and trained from scratch to output your exact categories.

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader, random_split
from torchvision.models import resnet50, ResNet50_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Use pretrained weights and their recommended transforms
weights = ResNet50_Weights.DEFAULT
transform = weights.transforms()

# Load full training dataset
full_trainset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)

classes = full_trainset.classes

# Split into train/validation
trainset, valset = random_split(
    full_trainset,
    [0.8, 0.2],
    generator=torch.Generator().manual_seed(42)
)

trainloader = DataLoader(trainset, batch_size=64, shuffle=True, num_workers=2)
valloader = DataLoader(valset, batch_size=64, shuffle=False, num_workers=2)

# Test dataset
testset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform
)
testloader = DataLoader(testset, batch_size=64, shuffle=False, num_workers=2)

print(f"Total training images: {len(full_trainset)}")
print(f"--> Split into: {len(trainset)} Train | {len(valset)} Validation")
print(f"Total testing images: {len(testset)}")

Using device: cuda
Total training images: 50000
--> Split into: 40000 Train | 10000 Validation
Total testing images: 10000


In [9]:
# Load pretrained model
model = resnet50(weights=weights)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Replace final classifier
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\amogh/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100.0%


In [11]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

In [12]:
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

In [13]:
phase1_epochs = 5

print("\nStarting Phase 1: Training only final FC layer\n")

for epoch in range(phase1_epochs):
    train_loss, train_acc = train_one_epoch(model, trainloader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, valloader, criterion, device)

    print(f"Phase 1 | Epoch [{epoch+1}/{phase1_epochs}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.2f}%\n")


Starting Phase 1: Training only final FC layer

Phase 1 | Epoch [1/5]
Train Loss: 0.6185 | Train Acc: 79.72%
Val   Loss: 0.6118 | Val   Acc: 79.19%

Phase 1 | Epoch [2/5]
Train Loss: 0.5627 | Train Acc: 81.16%
Val   Loss: 0.5941 | Val   Acc: 80.04%

Phase 1 | Epoch [3/5]
Train Loss: 0.5295 | Train Acc: 82.28%
Val   Loss: 0.5736 | Val   Acc: 80.55%

Phase 1 | Epoch [4/5]
Train Loss: 0.5105 | Train Acc: 82.73%
Val   Loss: 0.5666 | Val   Acc: 80.75%

Phase 1 | Epoch [5/5]
Train Loss: 0.4857 | Train Acc: 83.59%
Val   Loss: 0.5649 | Val   Acc: 80.67%



In [14]:
for param in model.layer3.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

fine_tune_optimizer = optim.Adam([
    {'params': model.layer3.parameters(), 'lr': 1e-6},
    {'params': model.layer4.parameters(), 'lr': 1e-5},
    {'params': model.fc.parameters(), 'lr': 1e-4}
])

print("Phase 2 setup complete. layer3, layer4, and fc are unfrozen.")

Phase 2 setup complete. layer3, layer4, and fc are unfrozen.


In [15]:
phase2_epochs = 5

print("\nStarting Phase 2: Fine tuning layer3, layer4, and FC\n")

for epoch in range(phase2_epochs):
    train_loss, train_acc = train_one_epoch(model, trainloader, criterion, fine_tune_optimizer, device)
    val_loss, val_acc = evaluate(model, valloader, criterion, device)

    print(f"Phase 2 | Epoch [{epoch+1}/{phase2_epochs}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.2f}%\n")

# --------------------------------------------------
# 12. Final Test Evaluation
# --------------------------------------------------
test_loss, test_acc = evaluate(model, testloader, criterion, device)

print("Final Test Performance")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")


Starting Phase 2: Fine tuning layer3, layer4, and FC

Phase 2 | Epoch [1/5]
Train Loss: 0.4214 | Train Acc: 85.78%
Val   Loss: 0.4753 | Val   Acc: 83.47%

Phase 2 | Epoch [2/5]
Train Loss: 0.3209 | Train Acc: 89.25%
Val   Loss: 0.4361 | Val   Acc: 84.93%

Phase 2 | Epoch [3/5]
Train Loss: 0.2488 | Train Acc: 91.86%
Val   Loss: 0.4232 | Val   Acc: 85.53%

Phase 2 | Epoch [4/5]
Train Loss: 0.1900 | Train Acc: 94.07%
Val   Loss: 0.4090 | Val   Acc: 85.93%

Phase 2 | Epoch [5/5]
Train Loss: 0.1440 | Train Acc: 95.74%
Val   Loss: 0.3931 | Val   Acc: 86.77%

Final Test Performance
Test Loss: 0.3943
Test Accuracy: 86.73%
